In [42]:
%matplotlib inline
import os, sys, glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles

# STEP 2: Create an PoseLandmarker object.
base_options = python.BaseOptions(model_asset_path='pose_landmarker.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True)
detector = vision.PoseLandmarker.create_from_options(options)

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("opencv:", cv2.__version__)
print("mediapipe:", mp.__version__)

def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks_list = detection_result.pose_landmarks
  annotated_image = np.copy(rgb_image)

  pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
  pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

  for pose_landmarks in pose_landmarks_list:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=pose_landmarks,
        connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
        landmark_drawing_spec=pose_landmark_style,
        connection_drawing_spec=pose_connection_style)

  return annotated_image

# --- Resolve image path (same idea as before) ---
# BASE = r"C:\Users\rs2476\SportsPose\data\Myvids\IMG_8384"
# SEARCH_EXTS = [".jpg", ".jpeg", ".png", ".heic", ".JPG", ".JPEG", ".PNG", ".HEIC"]
# candidates = [p for ext in SEARCH_EXTS for p in glob.glob(BASE + ext)]
# IMG_PATH = candidates[0] if candidates else BASE
IMG_PATH = r"C:\Users\rs2476\SportsPose\data\Myvids\IMG_6704.mov"

OUT_DIR = r"C:\Users\rs2476\SportsPose\results"
os.makedirs(OUT_DIR, exist_ok=True)

print("Resolved IMG_PATH:", IMG_PATH)

# mp_pose = mp.solutions.pose
# mp_drawing = mp.solutions.drawing_utils
# mp_styles = mp.solutions.drawing_styles

cap = cv2.VideoCapture(IMG_PATH)
assert cap is not None, f"Could not read video: {IMG_PATH}"

f = 1
joint_angles1 = []
joint_angles2 = []
joint_angles3 = []
joint_angles4 = []

while True:#f < 10:#
    # Read a frame
    ret, img_bgr = cap.read()
    # img_bgr = cv2.flip(img_bgr, -1)

    # Check if the frame was successfully read
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # # plt.figure(figsize=(8,6))
    # # plt.imshow(img_rgb)
    # # plt.title("MediaPipe Pose Landmarks")
    # # plt.axis("off")
    # # plt.show()
    
    # with mp_pose.Pose(
    #     static_image_mode=False,
    #     model_complexity=2,
    #     enable_segmentation=True,
    #     min_detection_confidence=0.5
    # ) as pose:
    #     results = pose.process(img_rgb)
    
    # # Check detection
    # # if not results.pose_landmarks:
    # #    raise RuntimeError("No pose detected by MediaPipe Pose.")
    
    # # Draw landmarks on a copy
    # overlay = img_rgb.copy()
    # mp_drawing.draw_landmarks(
    #     overlay,
    #     results.pose_landmarks,
    #     mp_pose.POSE_CONNECTIONS,
    #     landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style()
    # )
    # 
    # plt.figure(figsize=(8,6))
    # plt.imshow(overlay)
    # plt.title("MediaPipe Pose Landmarks")
    # plt.axis("off")
    # plt.show()

    img_rgb = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    
    results = detector.detect(img_rgb)
    
    # STEP 5: Process the detection result. In this case, visualize it.
    overlay = draw_landmarks_on_image(img_rgb.numpy_view(), results)
    
    out_img_path = os.path.join(OUT_DIR, f"mediapipe_pose_overlay_{f:04}.jpg")
    cv2.imwrite(out_img_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    print("Saved overlay to:", out_img_path)

    if results.pose_landmarks:    # MediaPipe pose has 33 landmarks
        # 2D in pixels
        coords2d = []
        for idx, landmark in enumerate(results.pose_landmarks[0]):
            coords2d.append([landmark.x * w, landmark.y * h, landmark.z * w, landmark.visibility])
        coords2d = np.array(coords2d, dtype=float)  # (33, 4)

         # 3D in pixels
        coords3d = []
        for idx, landmark in enumerate(results.pose_world_landmarks[0]):
            coords3d.append([landmark.x, landmark.y, landmark.z, landmark.visibility])
        coords2d = np.array(coords3d, dtype=float)  # (33, 4)
        
        # Output folder for tables
        OUTPUT_DIR = os.path.join(OUT_DIR, "outputMEDIAPIPE")
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        # MediaPipe 33 joint names (official order)
        JOINT_NAMES_33 = [
            "nose",                 # 0
            "left_eye_inner",       # 1
            "left_eye",             # 2
            "left_eye_outer",       # 3
            "right_eye_inner",      # 4
            "right_eye",            # 5
            "right_eye_outer",      # 6
            "left_ear",             # 7
            "right_ear",            # 8
            "mouth_left",           # 9
            "mouth_right",          # 10
            "left_shoulder",        # 11
            "right_shoulder",       # 12
            "left_elbow",           # 13
            "right_elbow",          # 14
            "left_wrist",           # 15
            "right_wrist",          # 16
            "left_pinky",           # 17
            "right_pinky",          # 18
            "left_index",           # 19
            "right_index",          # 20
            "left_thumb",           # 21
            "right_thumb",          # 22
            "left_hip",             # 23
            "right_hip",            # 24
            "left_knee",            # 25
            "right_knee",           # 26
            "left_ankle",           # 27
            "right_ankle",          # 28
            "left_heel",            # 29
            "right_heel",           # 30
            "left_foot_index",      # 31
            "right_foot_index"      # 32
        ]
        
        # ---- 2D sheet ----
        rows_2d = []
        for j in range(33):
            x_px, y_px, z_px, visibility = coords2d[j]
            rows_2d.append({
                "joint_index": j,
                "joint_name": JOINT_NAMES_33[j],
                "x_px": float(x_px),
                "y_px": float(y_px),
                "z_px": float(z_px),
                "visibility": float(visibility)
            })
        df2d = pd.DataFrame(rows_2d)
        
        # ---- 3D sheet ----
        rows_3d = []
        for j in range(33):
            x_px, y_px, z_px, visibility = coords3d[j]
            rows_3d.append({
                "joint_index": j,
                "joint_name": JOINT_NAMES_33[j],
                "x_px": float(x_px),
                "y_px": float(y_px),
                "z_px": float(z_px),
                "visibility": float(visibility)
            })
        df3d = pd.DataFrame(rows_3d)
        
        excel_path = os.path.join(OUTPUT_DIR, f"mediapipe_coordinates_2D_3D_{f:04}.xlsx")
        with pd.ExcelWriter(excel_path) as writer:
            df2d.to_excel(writer, sheet_name="2D_pixels", index=False)
            df3d.to_excel(writer, sheet_name="3D_world", index=False)
        
        print("Saved Excel with MediaPipe coordinates to:", excel_path)
        df2d.head()
    
        # Output folder
        OUTPUT_DIR = os.path.join(OUT_DIR, "outputMEDIAPIPE")
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        def angle_3d(A, B, C):
            A = np.asarray(A, float)
            B = np.asarray(B, float)
            C = np.asarray(C, float)
            BA = A - B
            BC = C - B
            dot = np.dot(BA, BC)
            norm = np.linalg.norm(BA) * np.linalg.norm(BC)
            if norm < 1e-8:
                return np.nan
            cos_t = np.clip(dot / norm, -1.0, 1.0)
            return float(np.degrees(np.arccos(cos_t)))
        
        def angle_2d(A, B, C):
            A = np.asarray(A[:2], float)
            B = np.asarray(B[:2], float)
            C = np.asarray(C[:2], float)
            BA = A - B
            BC = C - B
            dot = np.dot(BA, BC)
            norm = np.linalg.norm(BA) * np.linalg.norm(BC)
            if norm < 1e-8:
                return np.nan
            cos_t = np.clip(dot / norm, -1.0, 1.0)
            return float(np.degrees(np.arccos(cos_t)))
        
        # Build neighbor list from MediaPipe connections
        neighbors[11].add(13)
        neighbors[13].add(11)
        neighbors[13].add(15)
        neighbors[15].add(13)
        neighbors[12].add(14)
        neighbors[14].add(12)
        neighbors[14].add(16)
        neighbors[16].add(14)
        neighbors[23].add(25)
        neighbors[25].add(23)
        neighbors[25].add(27)
        neighbors[27].add(25)
        neighbors[24].add(26)
        neighbors[26].add(24)
        neighbors[26].add(28)
        neighbors[28].add(26)
        
        p2d = coords2d  # (33,2)
        p3d = coords3d  # (33,3)
        
        rows = []
        
        for center in range(33):
            nbrs = sorted(list(neighbors[center]))
            if len(nbrs) < 2:
                continue  # need at least 2 neighbors to form an angle
        
            # all unordered neighbor pairs (A,B,C where center is B)
            for i_idx in range(len(nbrs)):
                for j_idx in range(i_idx + 1, len(nbrs)):
                    iA = nbrs[i_idx]
                    iC = nbrs[j_idx]
                    iB = center
        
                    A2, B2, C2 = p2d[iA], p2d[iB], p2d[iC]
                    A3, B3, C3 = p3d[iA], p3d[iB], p3d[iC]
        
                    theta2 = angle_2d(A2, B2, C2)
                    theta3 = angle_3d(A3, B3, C3)
        
                    rows.append({
                        "center_index": iB,
                        "center_name": JOINT_NAMES_33[iB],
                        "jointA_index": iA,
                        "jointA_name": JOINT_NAMES_33[iA],
                        "jointC_index": iC,
                        "jointC_name": JOINT_NAMES_33[iC],
                        "angle_2D_deg": theta2,
                        "angle_3D_deg": theta3
                    })
    
        joint_angles1.append(rows[0])    # Left elbow
        joint_angles2.append(rows[1])    # Right elbow
        joint_angles3.append(rows[2])    # Left knee
        joint_angles4.append(rows[3])    # Right knee
        
        angles_all_df = pd.DataFrame(rows)
        
        angles_path = os.path.join(OUTPUT_DIR, f"mediapipe_angles_all_joints_{f:04}.xlsx")
        angles_all_df.to_excel(angles_path, index=False)
        
        print("Saved ALL joint angles (2D + 3D) to:", angles_path)
        angles_all_df.head()
    else:
        joint_angles1.append(rows[0])
        joint_angles2.append(rows[1])
        joint_angles3.append(rows[2])
        joint_angles4.append(rows[3])

    f = f + 1

cap.release()
cv2.destroyAllWindows()

joint_angles = pd.DataFrame(joint_angles1 + joint_angles2 + joint_angles3 + joint_angles4)

angles_path = os.path.join(OUTPUT_DIR, f"mediapipe_angles_all_joints_all.xlsx")
joint_angles.to_excel(angles_path, index=False)

print("Saved ALL joint angles (2D + 3D) to:", angles_path)
joint_angles.head()

Python: 3.13.5
numpy: 2.1.3
opencv: 4.13.0
mediapipe: 0.10.33
Resolved IMG_PATH: C:\Users\rs2476\SportsPose\data\Myvids\IMG_6704.mov
Saved overlay to: C:\Users\rs2476\SportsPose\results\mediapipe_pose_overlay_0001.jpg
Saved Excel with MediaPipe coordinates to: C:\Users\rs2476\SportsPose\results\outputMEDIAPIPE\mediapipe_coordinates_2D_3D_0001.xlsx
Saved ALL joint angles (2D + 3D) to: C:\Users\rs2476\SportsPose\results\outputMEDIAPIPE\mediapipe_angles_all_joints_0001.xlsx
Saved overlay to: C:\Users\rs2476\SportsPose\results\mediapipe_pose_overlay_0002.jpg
Saved Excel with MediaPipe coordinates to: C:\Users\rs2476\SportsPose\results\outputMEDIAPIPE\mediapipe_coordinates_2D_3D_0002.xlsx
Saved ALL joint angles (2D + 3D) to: C:\Users\rs2476\SportsPose\results\outputMEDIAPIPE\mediapipe_angles_all_joints_0002.xlsx
Saved overlay to: C:\Users\rs2476\SportsPose\results\mediapipe_pose_overlay_0003.jpg
Saved Excel with MediaPipe coordinates to: C:\Users\rs2476\SportsPose\results\outputMEDIAPIPE\me

,center_index,center_name,jointA_index,jointA_name,jointC_index,jointC_name,angle_2D_deg,angle_3D_deg
0,13,left_elbow,11,left_shoulder,15,left_wrist,176.907697,126.499328
1,13,left_elbow,11,left_shoulder,15,left_wrist,108.585742,120.280091
2,13,left_elbow,11,left_shoulder,15,left_wrist,106.581566,110.604357
3,13,left_elbow,11,left_shoulder,15,left_wrist,107.322218,110.690032
4,13,left_elbow,11,left_shoulder,15,left_wrist,107.957853,109.215582


In [37]:
neighbors

{0: set(),
 1: set(),
 2: set(),
 3: set(),
 4: set(),
 5: set(),
 6: set(),
 7: set(),
 8: set(),
 9: set(),
 10: set(),
 11: set(),
 12: set(),
 13: set(),
 14: set(),
 15: set(),
 16: set(),
 17: set(),
 18: set(),
 19: set(),
 20: set(),
 21: set(),
 22: set(),
 23: set(),
 24: set(),
 25: set(),
 26: set(),
 27: set(),
 28: set(),
 29: set(),
 30: set(),
 31: set(),
 32: set()}